# 01 - Dataset Inspection & Verification

**Project**: Intelligent Driver Activity Monitoring & Safety Agent  
**Dataset**: Driver Behavior Detection Using Smartphone - Dataset (Pawan Wawage & Yogesh Deshpande, *Data in Brief* / Mendeley Data)  
**Objective**: Inspect folder structure, verify file counts, row counts, sensor units, sampling frequencies, and data quality.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

possible_dirs = [
    Path("../dataset/raw"),
    Path("../../dataset/raw"),
    Path("driver-monitoring-agent/dataset/raw"),
    Path("dataset/raw"),
    Path(r"D:/driver ml model/driver-monitoring-agent/dataset/raw"),
]
base_dir = next((p for p in possible_dirs if p.exists()), None)
if base_dir is None:
    raise FileNotFoundError("Could not locate dataset/raw directory")
print(f"Base dataset directory located at: {base_dir.resolve()}")


## 1. Directory Structure: Multi-Driver vs Single-Driver Collections
- **`drivers/` (Table 3 in paper)**: Data from 7 distinct drivers (`Driver-1` to `Driver-7`).
- **`daywise/` (Table 4 in paper)**: Data from a **single specific driver** completing 2-way trips across 7 days (`Day-1S/1R` to `Day-7S`).


In [ ]:
driver_dirs = sorted(list((base_dir / "drivers").glob("Driver-*")))
print(f"Found {len(driver_dirs)} driver directories (Table 3):", [d.name for d in driver_dirs])

daywise_dirs = sorted(list((base_dir / "daywise" / "Daywise data").glob("Day-*")))
print(f"Found {len(daywise_dirs)} daywise trip directories for single driver (Table 4):", [d.name for d in daywise_dirs])


## 2. Sensor Files and Verified Record Counts
Inspect columns, row counts, and data types across all sensor streams.


In [ ]:
records = []
for d in driver_dirs:
    for s in ["Accelerometer.csv", "Gyroscope.csv", "Pressure.csv"]:
        p = d / s
        if p.exists():
            size_kb = round(p.stat().st_size / 1024, 1)
            df_head = pd.read_csv(p, nrows=2)
            row_count = sum(1 for _ in open(p, "rb")) - 1
            records.append({
                "Collection": "drivers (Table 3)",
                "Folder": d.name,
                "Sensor": s.replace(".csv", ""),
                "Rows": row_count,
                "Size_KB": size_kb,
                "Columns": list(df_head.columns)
            })
df_summary = pd.DataFrame(records)
print(df_summary.head(10))
print(f"Subtotal rows in drivers collection: {df_summary['Rows'].sum():,}")


## 3. Sensor Units and Coordinate Interpretations
- **Accelerometer**: $X, Y, Z$ in $\text{m/s}^2$ (norm $\|a\| \approx 9.81\ \text{m/s}^2$).
- **Gyroscope**: $X, Y, Z$ in $\text{rad/s}$ (Android standard sensor format; values centered around 0 with dynamic turns between 0.3 and 1.5 rad/s).
- **Orientation Limitation**: Smartphone placed horizontally in vehicle utility box, but vehicle-aligned coordinate orientation is uncalibrated in the dataset.


In [ ]:
p_accel = base_dir / "drivers" / "Driver-1" / "Accelerometer.csv"
p_gyro = base_dir / "drivers" / "Driver-1" / "Gyroscope.csv"
df_acc = pd.read_csv(p_accel, nrows=2000)
df_gyr = pd.read_csv(p_gyro, nrows=2000)

acc_mag = np.sqrt(df_acc["X"]**2 + df_acc["Y"]**2 + df_acc["Z"]**2)
gyr_mag = np.sqrt(df_gyr["X"]**2 + df_gyr["Y"]**2 + df_gyr["Z"]**2)
print(f"Accelerometer Norm (m/s^2) - Mean: {acc_mag.mean():.3f}, 95th%: {acc_mag.quantile(0.95):.3f}")
print(f"Gyroscope Norm (rad/s)     - Mean: {gyr_mag.mean():.4f}, 95th%: {gyr_mag.quantile(0.95):.4f}")


## 4. Sampling Frequency Analysis per Sensor Modality
Compute sampling intervals independently for Accelerometer, Gyroscope, Pressure, and Proximity.


In [ ]:
for sensor_name, p_sensor in [
    ("Accelerometer", base_dir / "drivers" / "Driver-1" / "Accelerometer.csv"),
    ("Gyroscope", base_dir / "drivers" / "Driver-1" / "Gyroscope.csv"),
    ("Pressure", base_dir / "drivers" / "Driver-3" / "Pressure.csv"),
    ("Proximity", base_dir / "daywise" / "Daywise data" / "Day-1R" / "Proximity.csv")
]:
    if p_sensor.exists():
        df_test = pd.read_csv(p_sensor, nrows=3000)
        diffs = df_test["Milliseconds"].diff()
        valid = diffs[(diffs > 0) & (diffs < 1000)]
        med_dt = valid.median()
        print(f"{sensor_name:15s} | Median dt: {med_dt:5.1f} ms | Effective Rate: {1000/med_dt:5.1f} Hz")


## 5. Corrupted & Anomaly Verification
- Confirm empty 0-byte GPS files.
- Confirm Driver-1 pressure sensor zero-reading anomaly.
- Confirm aborted trial Day-4E1 (only 1-2 rows).


In [ ]:
p_press = base_dir / "drivers" / "Driver-1" / "Pressure.csv"
df_press = pd.read_csv(p_press, nrows=500)
is_all_zero = (df_press["Millibars"] == 0).all()
print(f"Driver-1 Pressure All Zeroes: {is_all_zero} (Hardware/Permission issue)")

gps_sample = base_dir / "daywise" / "Daywise data" / "Day-1R" / "GPS.csv"
print(f"Day-1R GPS file size: {gps_sample.stat().st_size} bytes (Empty)")

e1_acc = base_dir / "daywise" / "Daywise data" / "Day-4E1" / "Accelerometer.csv"
print(f"Day-4E1 Accelerometer rows: {sum(1 for _ in open(e1_acc, 'rb')) - 1} (Aborted session)")
